# 8.0 Menu Analysis Overview

- This notebook analyzes Toast product mix data to understand menu-level revenue, volume, and profitability.
- The objective is to identify which items drive sales performance and which items may need pricing, promotion, or operational adjustments.
- This analysis uses item-level product mix exports across available quarters:
  - 2024 Q3
  - 2024 Q4
  - 2025 Q1
  - 2025 Q2
  - 2025 Q3
- Key focus areas include:
  - item revenue contribution
  - quantity sold
  - average price
  - gross profit
  - gross margin
  - voids, refunds, and waste where relevant
- This notebook builds toward menu engineering classifications such as:
  - high-volume / high-profit items
  - high-volume / low-profit items
  - low-volume / high-profit items
  - low-volume / low-profit items
- The goal is to translate menu performance into actionable recommendations for revenue growth and profitability.

## 8.1 Item-Level Dataset Construction

- Load item-level product mix data across all available quarters.
- Combine into a single unified dataset for analysis.
- Add year and quarter identifiers to enable time-based insights.
- This dataset will serve as the foundation for all menu analysis and engineering.

In [39]:
import pandas as pd
import os

# ✅ CORRECT base path (fixed typo)
base_path = "data/raw/orders_items_raw"

files = [
    # 2024
    ("2024/product_mix_2024_q3/items_2024_q3.csv", "2024", "Q3"),
    ("2024/product_mix_2024_q4/items_2024_q4.csv", "2024", "Q4"),

    # 2025
    ("2025/product_mix_2025_q1/items_2025_q1.csv", "2025", "Q1"),
    ("2025/product_mix_2025_q2/items_2025_q2.csv", "2025", "Q2"),
    ("2025/product_mix_2025_q3/items_2025_q3.csv", "2025", "Q3"),
]

dfs = []

for file_path, year, quarter in files:
    full_path = os.path.join(base_path, file_path)
    
    print(f"Loading: {full_path}")  # debug line
    
    df = pd.read_csv(full_path)
    
    df["year"] = year
    df["quarter"] = quarter
    
    dfs.append(df)

# Combine all quarters
items_combined_df = pd.concat(dfs, ignore_index=True)

# Preview
items_combined_df.head()

Loading: data/raw/orders_items_raw/2024/product_mix_2024_q3/items_2024_q3.csv
Loading: data/raw/orders_items_raw/2024/product_mix_2024_q4/items_2024_q4.csv
Loading: data/raw/orders_items_raw/2025/product_mix_2025_q1/items_2025_q1.csv
Loading: data/raw/orders_items_raw/2025/product_mix_2025_q2/items_2025_q2.csv
Loading: data/raw/orders_items_raw/2025/product_mix_2025_q3/items_2025_q3.csv


,masterId,parentId,itemGuid,Item,Sales Category,Item tags,Deferred,Qty sold,Avg. price,Item COGS,...,Gross margin (%),Tax,Waste count,Waste amount,Voided gross sale,Voided qty sold,Item qty incl. voids,Gross amount incl. voids,year,quarter
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7192.5,6.00,NaN,...,NaN,4306.17,0.0,0.0,344.33,47.5,7240.0,43490.47,2024,Q3
1,1.200000e+18,1.200000e+18,164a9725-b742-4fef-9325-19eb6fa4134c,Lunch Taco,Food,[],False,13.0,7.63,NaN,...,NaN,9.96,0.0,0.0,0.00,0.0,13.0,99.24,2024,Q3
2,1.200000e+18,1.200000e+18,06b361e0-15b2-45c4-9c38-b30bb3ad4a19,Lunch Enchilada,Food,[],False,27.0,7.03,NaN,...,NaN,19.07,0.0,0.0,0.00,0.0,27.0,189.75,2024,Q3
3,1.200000e+18,1.200000e+18,5c7d285b-38ec-4637-a0b2-60d9d706169f,Lunch Tamale,Food,[],False,8.0,7.00,NaN,...,NaN,5.62,0.0,0.0,0.00,0.0,8.0,56.00,2024,Q3
4,1.200000e+18,1.200000e+18,fdb31449-e5ab-46af-a22d-4dfc88ece538,Lunch Tostada,Food,[],False,3.0,7.08,NaN,...,NaN,2.14,0.0,0.0,0.00,0.0,3.0,21.25,2024,Q3


## 8.2 Column Cleaning & Standardization

- Standardize column names for consistency and usability.
- Convert all column names to lowercase and replace spaces with underscores.
- Ensure key numerical fields are correctly typed for analysis.
- Select and retain only relevant columns for menu engineering.
- This step prepares the dataset for accurate aggregation and analysis.

In [40]:
# Standardize column names (UPDATED — removes parentheses)
items_combined_df.columns = (
    items_combined_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(".", "")
    .str.replace("(", "")
    .str.replace(")", "")
)

# Preview updated columns
items_combined_df.columns

Index(['masterid', 'parentid', 'itemguid', 'item', 'sales_category',
       'item_tags', 'deferred', 'qty_sold', 'avg_price', 'item_cogs',
       'gross_sales', 'discount_amount', 'refund_amount', 'void_amount',
       'net_sales', 'cogs', 'gross_profit', 'gross_margin_%', 'tax',
       'waste_count', 'waste_amount', 'voided_gross_sale', 'voided_qty_sold',
       'item_qty_incl_voids', 'gross_amount_incl_voids', 'year', 'quarter'],
      dtype='object')

In [41]:
# Select core columns for analysis
items_clean_df = items_combined_df[[
    "item",
    "sales_category",
    "qty_sold",
    "avg_price",
    "net_sales",
    "year",
    "quarter",
    "gross_profit",
    "gross_margin_%"
]].copy()

items_clean_df.head()

,item,sales_category,qty_sold,avg_price,net_sales,year,quarter,gross_profit,gross_margin_%
0,NaN,NaN,7192.5,6.00,43094.44,2024,Q3,NaN,NaN
1,Lunch Taco,Food,13.0,7.63,99.24,2024,Q3,NaN,NaN
2,Lunch Enchilada,Food,27.0,7.03,189.75,2024,Q3,NaN,NaN
3,Lunch Tamale,Food,8.0,7.00,56.00,2024,Q3,NaN,NaN
4,Lunch Tostada,Food,3.0,7.08,21.25,2024,Q3,NaN,NaN


In [42]:
# Ensure numeric fields are properly typed
numeric_cols = ["qty_sold", "avg_price", "net_sales", "gross_profit", "gross_margin_%"]

for col in numeric_cols:
    items_clean_df[col] = pd.to_numeric(items_clean_df[col], errors="coerce")

items_clean_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 991 entries, 0 to 990
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   item            986 non-null    object 
 1   sales_category  986 non-null    object 
 2   qty_sold        991 non-null    float64
 3   avg_price       991 non-null    float64
 4   net_sales       991 non-null    float64
 5   year            991 non-null    object 
 6   quarter         991 non-null    object 
 7   gross_profit    0 non-null      float64
 8   gross_margin_%  0 non-null      float64
dtypes: float64(5), object(4)
memory usage: 69.8+ KB


## 8.3 Item Revenue & Volume Ranking

- Identify top-performing items based on total revenue and quantity sold.
- Rank items to determine which products drive the majority of business performance.
- This analysis highlights:
  - revenue leaders (high total sales)
  - volume leaders (most frequently ordered items)
- These insights form the foundation for menu optimization decisions.

In [43]:
# Aggregate item performance across all quarters
item_summary_df = (
    items_clean_df
    .groupby("item", as_index=False)
    .agg(
        total_qty_sold=("qty_sold", "sum"),
        total_revenue=("net_sales", "sum"),
        avg_price=("avg_price", "mean")
    )
)

item_summary_df.head()

,item,total_qty_sold,total_revenue,avg_price
0,"1 One Taco, One Enchilada, and One Chalupa",46.0,480.25,10.420
1,"10 One Enchilada, One Burrito & One Chile Relleno",53.0,559.48,10.638
2,"11 One Beef Burrito, One Enchilada & One Tamale",104.0,1098.23,10.554
3,"12 One Burrito, One Enchilada Rice & Beans",76.0,811.18,10.850
4,"13 One Chile Relleno, One Taco, Rice & Beans",45.0,466.25,10.332


In [44]:
top_revenue_items = (
    item_summary_df
    .sort_values("total_revenue", ascending=False)
    .head(10)
)

top_revenue_items

,item,total_qty_sold,total_revenue,avg_price
129,Nelson Special,3152.0,32397.66,10.302
24,Burrito California,2299.0,26369.21,11.506
177,Sweet tea,10353.0,25886.26,2.504
165,Small Cheese Dip,4836.0,20559.64,4.258
80,Large Cheese Dip Take-out,1706.0,16651.55,9.782
207,small cheesedip takeout,3891.0,16594.22,4.266
105,Lunch Nelson Special,1687.0,13943.26,8.298
79,Large Cheese Dip,1871.0,13571.62,7.260
62,Fajitas Steak,920.0,12233.18,13.326
176,Street Tacos,1279.0,12141.80,9.542


In [45]:
top_volume_items = (
    item_summary_df
    .sort_values("total_qty_sold", ascending=False)
    .head(10)
)

top_volume_items

,item,total_qty_sold,total_revenue,avg_price
177,Sweet tea,10353.0,25886.26,2.504
208,water,6817.0,0.00,0.000
165,Small Cheese Dip,4836.0,20559.64,4.258
207,small cheesedip takeout,3891.0,16594.22,4.266
129,Nelson Special,3152.0,32397.66,10.302
20,Beef Taco,2994.0,7849.36,2.638
51,Coke,2831.0,7086.56,2.506
124,Mr. Pibb,2727.0,6828.84,2.508
24,Burrito California,2299.0,26369.21,11.506
79,Large Cheese Dip,1871.0,13571.62,7.260


## 8.3.1 SQL Item Performance Analysis (Raw Data)

- Rebuild item-level revenue and volume analysis using SQL
- Validate Python aggregation results with SQL queries
- Demonstrate ability to analyze transactional item-level data using SQL

- Objective:
    - Identify top revenue-generating items
    - Identify highest volume items
    - Compare revenue vs volume dynamics
- Note: Raw item names include inconsistencies (e.g., duplicates, formatting issues)
- This demonstrates how unclean data can distort performance insights

In [46]:
import duckdb

con = duckdb.connect()
con.register("items_clean_df", items_clean_df)

con.execute("""
SELECT
    item,
    SUM(qty_sold) AS total_qty_sold,
    SUM(net_sales) AS total_revenue,
    ROUND(AVG(avg_price), 2) AS avg_price
FROM items_clean_df
GROUP BY item
ORDER BY total_revenue DESC
LIMIT 10
""").fetchdf()

,item,total_qty_sold,total_revenue,avg_price
0,None,103469.0,611652.17,5.92
1,Nelson Special,3152.0,32397.66,10.30
2,Burrito California,2299.0,26369.21,11.51
3,Sweet tea,10353.0,25886.26,2.50
4,Small Cheese Dip,4836.0,20559.64,4.26
5,Large Cheese Dip Take-out,1706.0,16651.55,9.78
6,small cheesedip takeout,3891.0,16594.22,4.27
7,Lunch Nelson Special,1687.0,13943.26,8.30
8,Large Cheese Dip,1871.0,13571.62,7.26
9,Fajitas Steak,920.0,12233.18,13.33


## 8.3.2 SQL Item Ranking (Window Function)

- Rank items based on total revenue using SQL window functions
- Identify top-performing menu items

- Objective:
    - Highlight highest impact items
    - Demonstrate advanced SQL capability (window functions)

In [47]:
con.execute("""
SELECT
    item,
    SUM(net_sales) AS total_revenue,
    RANK() OVER (ORDER BY SUM(net_sales) DESC) AS revenue_rank
FROM items_clean_df
GROUP BY item
ORDER BY revenue_rank
LIMIT 10
""").fetchdf()

,item,total_revenue,revenue_rank
0,None,611652.17,1
1,Nelson Special,32397.66,2
2,Burrito California,26369.21,3
3,Sweet tea,25886.26,4
4,Small Cheese Dip,20559.64,5
5,Large Cheese Dip Take-out,16651.55,6
6,small cheesedip takeout,16594.22,7
7,Lunch Nelson Special,13943.26,8
8,Large Cheese Dip,13571.62,9
9,Fajitas Steak,12233.18,10


### SQL Insight Translation

- A small number of items dominate total revenue, confirming a Pareto-like distribution
- High-ranking items should be:
    - protected (quality + consistency)
    - promoted (menu placement)
    - leveraged for bundling strategies

- SQL confirms:
    - revenue concentration is driven by a small set of core menu items
    - add-on items (e.g., cheese dip, beverages) play a key role in increasing AOV

- Business Implication:
    - Revenue growth should focus on amplifying high-performing items rather than expanding the menu
    - Strategic bundling and upselling should center around top-ranked items


## 8.4 Item Performance Insights

- Revenue is driven by a mix of high-volume, low-price items and mid-to-high price core menu items.

### Key Observations

- **Top Revenue Drivers**
  - Nelson Special and Burrito California lead total revenue.
  - These items combine strong volume with higher price points.

- **High Volume, Low Price Items**
  - Sweet tea is the highest volume item by a large margin.
  - Other beverages (Coke, Mr. Pibb) also show high volume but low revenue contribution per unit.
  - These items drive traffic but not significant revenue individually.

- **Strong Mid-Tier Performers**
  - Cheese dips (small and large) show both high volume and meaningful revenue.
  - These are likely strong add-on items that increase overall order value.

- **Menu Duplication / Data Quality Issue**
  - Items like:
    - "Small Cheese Dip"
    - "small cheesedip takeout"
  - Likely represent the same product but are split due to naming inconsistencies.
  - This fragments true performance and should be standardized.

- **Zero-Revenue High-Volume Item**
  - "water" appears with high volume but zero revenue.
  - This likely represents free items and should be excluded from revenue analysis.

### Strategic Implications

- Core revenue is driven by:
  - a few high-performing entrees
  - supported by high-volume add-ons and beverages

- Opportunities exist to:
  - standardize item naming
  - optimize pricing on high-volume low-price items
  - promote high-performing add-ons to increase AOV

## 8.5 Item Name Standardization

- Normalize item names to ensure consistent aggregation and accurate analysis.
- Identify duplicate items caused by naming variations (e.g., case differences, spacing, suffixes like “takeout”).
- Create a standardized item name field to consolidate performance metrics.
- This step ensures that true item performance is not fragmented across multiple labels.

In [48]:
# Improved standardization
# ==============================
# 8.5 FINAL ITEM STANDARDIZATION
# ==============================

# Step 1: Base cleaning
items_clean_df["item_standardized"] = (
    items_clean_df["item"]
    .str.lower()
    .str.strip()
    
    # remove takeout variations
    .str.replace("takeout", "", regex=False)
    .str.replace("take out", "", regex=False)
    .str.replace("to go", "", regex=False)
    .str.replace("togo", "", regex=False)
    
    # fix wording issues
    .str.replace("cheesedip", "cheese dip", regex=False)
    
    # normalize formatting
    .str.replace("-", " ", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# ==============================
# Step 2: Manual Mapping (FINAL FIX)
# ==============================

items_clean_df["item_standardized"] = items_clean_df["item_standardized"].replace({
    "large cheese dip take out": "large cheese dip",
    "large cheese dip  ": "large cheese dip",
    "small cheese dip  ": "small cheese dip"
})

# ==============================
# Step 3: Rebuild Clean Summary
# ==============================

item_summary_clean_df = (
    items_clean_df
    .groupby("item_standardized", as_index=False)
    .agg(
        total_qty_sold=("qty_sold", "sum"),
        total_revenue=("net_sales", "sum"),
        avg_price=("avg_price", "mean")
    )
)

# ==============================
# Step 4: Top Revenue (Clean)
# ==============================

top_revenue_items_clean = (
    item_summary_clean_df
    .sort_values("total_revenue", ascending=False)
    .head(10)
)

top_revenue_items_clean

,item_standardized,total_qty_sold,total_revenue,avg_price
179,small cheese dip,8727.0,37153.86,4.262
139,nelson special,3152.0,32397.66,10.302
88,large cheese dip,3577.0,30223.17,8.521
25,burrito california,2299.0,26369.21,11.506
191,sweet tea,10353.0,25886.26,2.504
115,lunch nelson special,1687.0,13943.26,8.298
66,fajitas steak,920.0,12233.18,13.326
190,street tacos,1279.0,12141.80,9.542
47,chimichanga,1124.0,11976.42,10.660
62,fajita buenavista,821.0,11392.19,13.934


In [49]:
# Check for duplicates after standardization
duplicate_check = (
    items_clean_df
    .groupby("item_standardized")["item"]
    .nunique()
    .sort_values(ascending=False)
)

duplicate_check.head(20)

item_standardized
small cheese dip                              2
large cheese dip                              2
melow yellow                                  1
mexican lasagna                               1
mexican rice                                  1
milk 16 oz                                    1
mr. pibb                                      1
mushroom quesadilla                           1
nachos                                        1
nachos el carbon                              1
nachos machos                                 1
nelson special                                1
order ground beef                             1
1 one taco, one enchilada, and one chalupa    1
order of grilled chicken                      1
order of grilled shrimp (8)                   1
order of grilled steak                        1
order of lettuce 3oz                          1
order shredded chicken                        1
panchito cheeseburger and fries               1
Name: item, dtype: int

In [50]:
# Rebuild item summary using standardized names
item_summary_clean_df = (
    items_clean_df
    .groupby("item_standardized", as_index=False)
    .agg(
        total_qty_sold=("qty_sold", "sum"),
        total_revenue=("net_sales", "sum"),
        avg_price=("avg_price", "mean")
    )
)

item_summary_clean_df.head()

,item_standardized,total_qty_sold,total_revenue,avg_price
0,"1 one taco, one enchilada, and one chalupa",46.0,480.25,10.420
1,"10 one enchilada, one burrito & one chile relleno",53.0,559.48,10.638
2,"11 one beef burrito, one enchilada & one tamale",104.0,1098.23,10.554
3,"12 one burrito, one enchilada rice & beans",76.0,811.18,10.850
4,"13 one chile relleno, one taco, rice & beans",45.0,466.25,10.332


## 8.5.1 SQL Item Performance (Standardized)

- Re-run SQL analysis using standardized item names
- Ensure accurate aggregation of revenue and volume
- Eliminate duplication caused by inconsistent naming

- Objective:
    - Reflect true item performance
    - Improve reliability of revenue insights
    - Align SQL analysis with cleaned dataset

In [51]:
con.unregister("items_clean_df")
con.register("items_clean_df", items_clean_df)

In [52]:
con.execute("""
SELECT
    item_standardized,
    SUM(qty_sold) AS total_qty_sold,
    SUM(net_sales) AS total_revenue,
    ROUND(AVG(avg_price), 2) AS avg_price
FROM items_clean_df
GROUP BY item_standardized
ORDER BY total_revenue DESC
LIMIT 10
""").fetchdf()

,item_standardized,total_qty_sold,total_revenue,avg_price
0,None,103469.0,611652.17,5.92
1,small cheese dip,8727.0,37153.86,4.26
2,nelson special,3152.0,32397.66,10.30
3,large cheese dip,3577.0,30223.17,8.52
4,burrito california,2299.0,26369.21,11.51
5,sweet tea,10353.0,25886.26,2.50
6,lunch nelson special,1687.0,13943.26,8.30
7,fajitas steak,920.0,12233.18,13.33
8,street tacos,1279.0,12141.80,9.54
9,chimichanga,1124.0,11976.42,10.66


## 8.6 Cleaned Item Rankings (Standardized)

- Recalculate top-performing items using standardized item names.
- This ensures accurate aggregation of revenue and volume.
- Compare results against the original rankings to identify changes caused by data cleaning.
- This provides a true view of item performance.

In [53]:
# Top revenue items (cleaned)
top_revenue_items_clean = (
    item_summary_clean_df
    .sort_values("total_revenue", ascending=False
                )
    .head(20)
)

top_revenue_items_clean

,item_standardized,total_qty_sold,total_revenue,avg_price
179,small cheese dip,8727.0,37153.86,4.262
139,nelson special,3152.0,32397.66,10.302
88,large cheese dip,3577.0,30223.17,8.521
25,burrito california,2299.0,26369.21,11.506
191,sweet tea,10353.0,25886.26,2.504
115,lunch nelson special,1687.0,13943.26,8.298
66,fajitas steak,920.0,12233.18,13.326
190,street tacos,1279.0,12141.80,9.542
47,chimichanga,1124.0,11976.42,10.660
62,fajita buenavista,821.0,11392.19,13.934


## 8.6.1 Revenue Distribution (Top Items)

- Visualize revenue contribution of top-performing items.
- This helps highlight concentration of revenue across the menu.
- Supports identification of core revenue drivers.

In [ ]:
import matplotlib.pyplot as plt

top_revenue_items_clean = (
    item_summary_clean_df
    .sort_values("total_revenue", ascending=False)
    .head(10)
)

plt.figure()

plt.bar(
    top_revenue_items_clean["item_standardized"],
    top_revenue_items_clean["total_revenue"]
)

plt.xticks(rotation=45)
plt.xlabel("Item")
plt.ylabel("Total Revenue")
plt.title("Top 10 Items by Revenue")

plt.tight_layout()
plt.show()

## 8.7 Final Menu Insights & Strategic Recommendations

- This analysis identifies the core drivers of revenue and customer behavior within the menu.
- After cleaning and standardizing item data, true item performance becomes visible.

### Key Findings

- **Revenue Leaders**
  - Small cheese dip emerges as the top revenue-generating item after consolidation.
  - Nelson Special and Burrito California are the strongest entrée-level revenue drivers.
  - These items represent the foundation of total sales performance.

- **High-Volume Drivers**
  - Sweet tea dominates volume but contributes less revenue per unit due to low pricing.
  - Beverages and low-cost items act as traffic drivers rather than revenue maximizers.

- **AOV (Average Order Value) Drivers**
  - Cheese dips (small and large) function as strong add-on items.
  - These items significantly increase order value when attached to main dishes.
  - Their high volume and consistent pricing make them ideal upsell targets.

- **Menu Structure Insight**
  - Revenue is not evenly distributed across the menu.
  - A small number of items drive a disproportionate share of total revenue.
  - This aligns with a typical Pareto distribution in restaurant sales.

---

### Strategic Recommendations

#### 1. Protect and Promote Top Revenue Items
- Maintain quality and consistency for:
  - Nelson Special
  - Burrito California
  - Cheese dips
- Highlight these items on menus and ordering interfaces.

---

#### 2. Leverage Add-On Strategy to Increase AOV
- Position cheese dips as default add-ons:
  - “Add a cheese dip for $X”
- Train staff (or configure POS prompts) to upsell these items consistently.
- Bundle dips with entrees to increase conversion.

---

#### 3. Optimize Pricing for High-Volume Items
- Evaluate small price increases on:
  - sweet tea
  - soft drinks
- Even minor adjustments can significantly impact total revenue due to high volume.

---

#### 4. Simplify and Standardize Menu Naming
- Eliminate duplicate item naming across POS system.
- Ensure consistency between:
  - dine-in
  - takeout
  - variations
- This improves reporting accuracy and operational clarity.

---

#### 5. Focus on Core Menu Performance
- Identify underperforming items outside top contributors.
- Consider:
  - repositioning
  - repricing
  - removal if necessary
- Keep the menu focused on high-performing items.

---

### Business Impact

- Revenue growth can be achieved without increasing customer traffic.
- Improvements come from:
  - better item positioning
  - pricing optimization
  - upsell strategy
- This approach maximizes value from existing demand.

---

### Final Takeaway

- The menu already contains strong revenue drivers.
- The opportunity lies in:
  - amplifying what works
  - eliminating inefficiencies
  - improving execution at the point of sale

- With minimal operational changes, this business can increase revenue through smarter menu design and positioning.

 ## 8.8 Category Performance Analysis

- Analyze performance within each menu category to identify what truly drives revenue.
- Instead of only comparing high-level categories (e.g., Food vs Beverage), this analysis focuses on:
  - top-performing items within each category
  - revenue concentration inside categories

- This helps answer:
  “Which specific items are responsible for category-level performance?”

- This provides a more actionable view than category totals alone.

## 8.8 Category Performance Analysis

- Analyze performance within each menu category to identify what truly drives revenue.
- Instead of only comparing high-level categories (e.g., Food vs Beverage), this analysis focuses on:
  - top-performing items within each category
  - revenue concentration inside categories

- This helps answer:
  “Which specific items are responsible for category-level performance?”

- This provides a more actionable view than category totals alone.

In [ ]:
top_items_by_category = (
    items_clean_df
    .groupby(["sales_category", "item"], as_index=False)
    .agg(total_revenue=("net_sales", "sum"))
)

top_items_by_category = (
    top_items_by_category
    .sort_values(["sales_category", "total_revenue"], ascending=[True, False])
    .groupby("sales_category")
    .head(5)
)

top_items_by_category

In [ ]:
import matplotlib.pyplot as plt

food_items = top_items_by_category[
    top_items_by_category["sales_category"] == "Food"
]

plt.figure()

plt.bar(
    food_items["item"],
    food_items["total_revenue"]
)

plt.xticks(rotation=45)
plt.xlabel("Item")
plt.ylabel("Total Revenue")
plt.title("Top Food Items by Revenue")

plt.tight_layout()
plt.show()

## 8.9 Category-Level Insights

- Category-level analysis shows that overall performance is driven by a small number of high-performing items within each category.

### Key Observations

- **Revenue Concentration Within Categories**
  - A few core items dominate revenue within the Food category.
  - Items such as Nelson Special, Burrito California, and cheese dips are primary contributors.

- **Category Structure Insight**
  - The Food category is not evenly distributed—performance is concentrated in specific items rather than across the full menu.
  - This reinforces the Pareto pattern observed earlier.

- **Beverage Category Behavior**
  - Beverage items show high volume but lower revenue per item.
  - Revenue contribution is spread across multiple items rather than dominated by one.

---

### Strategic Implications

- Focus optimization efforts on top-performing items within each category rather than the category as a whole.
- Promote high-performing items more aggressively in menus and ordering systems.
- Use beverage items as consistent add-on opportunities to increase AOV.
- Evaluate low-performing items within each category for simplification or removal.

---

### Final Takeaway

- Categories do not drive revenue—items within categories do.
- Business performance is shaped by a small number of high-impact products.
- Optimizing these key items provides the greatest opportunity for revenue growth.

## 8.10 SQL Order Value Distribution (Segmentation Analysis)

- Analyze how orders are distributed across value tiers using SQL
- Identify which segments contribute most to total revenue
- Support AOV optimization and pricing strategy

- SQL Techniques Used:
    - CASE WHEN (value segmentation)
    - GROUP BY aggregation
    - Window functions (revenue percentage)

In [24]:
import pandas as pd
import duckdb

orders_clean = pd.read_csv("../data/cleaned/orders_clean.csv")

# restore datetime (CSV limitation)
orders_clean["order_datetime"] = pd.to_datetime(orders_clean["order_datetime"])

con = duckdb.connect()
con.register("orders_clean", orders_clean)

In [25]:
con.execute("""
SELECT
    CASE
        WHEN order_total < 15 THEN 'Low Value (<$15)'
        WHEN order_total BETWEEN 15 AND 30 THEN 'Mid Value ($15–$30)'
        ELSE 'High Value ($30+)'
    END AS value_segment,

    COUNT(*) AS total_orders,
    SUM(order_total) AS total_revenue,
    ROUND(AVG(order_total), 2) AS avg_order_value

FROM orders_clean

GROUP BY value_segment
ORDER BY total_revenue DESC
""").fetchdf()


,value_segment,total_orders,total_revenue,avg_order_value
0,High Value ($30+),7031,316591.06,45.03
1,Mid Value ($15–$30),8701,197083.51,22.65
2,Low Value (<$15),6424,63617.09,9.90


In [26]:
con.execute("""
SELECT
    value_segment,
    total_revenue,
    ROUND(total_revenue / SUM(total_revenue) OVER (), 3) AS revenue_pct
FROM (
    SELECT
        CASE
            WHEN order_total < 15 THEN 'Low Value (<$15)'
            WHEN order_total BETWEEN 15 AND 30 THEN 'Mid Value ($15–$30)'
            ELSE 'High Value ($30+)'
        END AS value_segment,
        SUM(order_total) AS total_revenue
    FROM orders_clean
    GROUP BY value_segment
)
ORDER BY total_revenue DESC
""").fetchdf()

,value_segment,total_revenue,revenue_pct
0,High Value ($30+),316591.06,0.548
1,Mid Value ($15–$30),197083.51,0.341
2,Low Value (<$15),63617.09,0.110


In [27]:
con.execute("""
SELECT
    EXTRACT(HOUR FROM order_datetime) AS order_hour,
    COUNT(*) AS high_value_orders,
    SUM(order_total) AS total_revenue
FROM orders_clean
WHERE order_total > 30
GROUP BY order_hour
ORDER BY total_revenue DESC
""").fetchdf()

,order_hour,high_value_orders,total_revenue
0,17,1214,53127.07
1,12,1031,49328.88
2,18,1067,48490.70
3,11,1003,46836.01
4,16,858,36567.32
5,19,528,24455.35
6,15,481,21036.18
7,13,477,20704.32
8,14,361,15600.40
9,10,9,359.60


## 8.11 AOV vs Volume Tradeoff

- Compare order volume vs average order value across time.
- Identify whether revenue is driven by many small orders or fewer high-value orders.
- Highlights operational vs pricing-driven growth opportunities.

In [28]:
con.execute("""
SELECT
    EXTRACT(HOUR FROM order_datetime) AS order_hour,
    
    COUNT(*) AS total_orders,
    SUM(order_total) AS total_revenue,
    ROUND(AVG(order_total), 2) AS avg_order_value

FROM orders_clean

GROUP BY order_hour
ORDER BY total_orders DESC
""").fetchdf()

,order_hour,total_orders,total_revenue,avg_order_value
0,11,3841,92533.05,24.09
1,12,3259,86567.47,26.56
2,17,3224,90153.19,27.96
3,18,2838,81880.89,28.85
4,16,2517,66872.10,26.57
5,13,1841,42734.53,23.21
6,15,1617,40481.66,25.04
7,19,1560,42504.80,27.25
8,14,1381,32313.08,23.40
9,10,64,1024.07,16.00


## 8.12 Revenue Efficiency by Time

- Identify periods with the highest revenue per order.
- Highlights opportunities for AOV-focused strategies.
- Helps distinguish between high-volume vs high-efficiency periods.

In [29]:
con.execute("""
SELECT
    EXTRACT(HOUR FROM order_datetime) AS order_hour,
    
    ROUND(AVG(order_total), 2) AS avg_order_value,
    SUM(order_total) AS total_revenue

FROM orders_clean

GROUP BY order_hour
ORDER BY avg_order_value DESC
""").fetchdf()

,order_hour,avg_order_value,total_revenue
0,18,28.85,81880.89
1,17,27.96,90153.19
2,19,27.25,42504.80
3,16,26.57,66872.10
4,12,26.56,86567.47
5,15,25.04,40481.66
6,11,24.09,92533.05
7,14,23.40,32313.08
8,13,23.21,42734.53
9,20,17.04,221.57


## 8.13 Menu Strategy Insights

## Menu Strategy Insights

- Revenue is driven by higher-value orders, indicating that menu design should encourage larger ticket sizes rather than focusing solely on volume  

- Add-on items (e.g., chips + cheese dip, drinks, sides) should be positioned to naturally increase Average Order Value (AOV)  

- Bundling strategies can simplify decision-making and guide customers toward higher-value combinations  

- Pricing should be structured around key thresholds (e.g., $15, $25, $30) to encourage customers to spend more per order  

- Menu layout and promotion should align with peak demand periods to maximize revenue during high-traffic hours